# load_openaire_researchproduct

Prototipo del nodo `load_openaire_researchproduct` del pipeline `load_openaire`. No guarda datasets.


In [ ]:
from datetime import date
import pandas as pd

%load_ext kedro.ipython


In [ ]:
df_researchproduct_raw = catalog.load('raw/openaire/researchproduct/parquet/researchproduct_dev')
df_researchproduct_raw.head(2)


In [ ]:
def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


In [ ]:
def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openaire_researchproduct(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    expected_columns = [
        'id',
        'openAccessColor',
        'publiclyFunded',
        'type',
        'language',
        'country',
        'mainTitle',
        'description',
        'publicationDate',
        'format',
        'bestAccessRight',
        'indicators',
        'isGreen',
        'isInDiamondJournal',
        'publisher',
        'source',
        'container',
        'contributor',
        'contactPerson',
        'coverage',
        'contactPerson',
        'embargoEndDate',
        'dateOfCollection',        
        '_filter_param',
        '_filter_value',
        '_extract_datetime',
    ]

    # Agregar columnas faltantes con NaN
    for col in expected_columns:
        if col not in df.columns:
            df[col] = pd.NA

    df = df.convert_dtypes()

    df_researchproduct = df[expected_columns].copy()
    df_researchproduct.reset_index(drop=True, inplace=True)

    # language
    df_researchproduct['language'] = df_researchproduct['language'].apply(
        lambda x: x if isinstance(x, dict) else {}
    )
    df_researchproduct['language_code'] = df_researchproduct['language'].apply(lambda x: x['code'])
    df_researchproduct['language_label'] = df_researchproduct['language'].apply(lambda x: x['label'])

    
    ## bestAccessRight
    df_researchproduct['bestAccessRight_label'] = df['bestAccessRight'].apply(lambda x: x['label'] if x else None)
    df_researchproduct['bestAccessRight_scheme'] = df['bestAccessRight'].apply(lambda x: x['scheme'] if x else None)

    ## indicators
    df_indicators = pd.json_normalize(df['indicators']).reset_index(drop=True)
    
    indicators_expected_columns = [
        "citationImpact.citationClass",
        "citationImpact.citationCount",
        "citationImpact.impulse",
        "citationImpact.impulseClass",
        "citationImpact.influence",
        "citationImpact.influenceClass",
        "citationImpact.popularity",
        "citationImpact.popularityClass",
        "usageCounts.downloads",
        "usageCounts.views",
    ]

    # Agregar columnas para indicators y faltantes con NaN
    for col in indicators_expected_columns:
        if col not in df_indicators.columns:
            df_indicators[col] = pd.NA

    df_researchproduct = pd.concat([df_researchproduct.drop(columns=['indicators']).reset_index(drop=True), df_indicators], axis=1)

    # TODO country
    # TODO description
    # TODO format
    # TODO instance
    # TODO container
    # TODO contributor
    # TODO contactPerson
    # TODO coverage

    ## drop de columnas procesadas en otros df
    df_researchproduct.drop(columns=[
        'country', 'bestAccessRight', 
        'language', 'format',  
        'container', 'source', 'description',
        'contributor', 'contactPerson', 'coverage'
        ], inplace=True)

    df_researchproduct = _add_openaire_loaded_metadata(df_researchproduct)

    return df_researchproduct


In [ ]:
df_researchproduct = load_openaire_researchproduct(df_researchproduct_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_researchproduct', 'rows': len(df_researchproduct), 'columns': len(df_researchproduct.columns)}])


In [ ]:
df_researchproduct.head(2)
